<a href="https://colab.research.google.com/github/leonardo-eid/Analise-Desempenho-Escolar_ICMC-Jr/blob/main/modelagemPreditiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MODELAGEM PREDITIVA**

## 1. É possível prever alguma das seguintes variáveis alvo: "final score", "grade", "pass fail"?

In [ ]:
### FUNÇÃO PARA CARREGAMENTO E LIMPEZA DOS DADOS
def carregamentoELimpeza():
  import pandas as pd

  # Setup e carregamento
  url = 'https://raw.githubusercontent.com/leonardo-eid/Analise-Desempenho-Escolar_ICMC-Jr/refs/heads/main/data/student_performance_interactions.csv'
  df = pd.read_csv(url)

  # Removendo o ID
  df.drop('student_id', axis=1, inplace=True)

  # Removendo linhas com erros
  colunas_verificar = ['math_prev_score', 'science_prev_score', 'language_prev_score', 'homework_completion_rate']
  for col in colunas_verificar:
      df = df[df[col] <= 100]

  return df

## Variável "Final_Score"

### Pré-processamento dos dados

Nessa primeira estapa, vamos carregar os dados, fazer a limpeza, transformar as variáveis qualitativas em quantitativas e remover colunas que atrapalham na predição

In [ ]:
# Aplicamos a função
df = carregamentoELimpeza()

# Transformação de variáveis textuais para numéricas
# mapeamento ordinal para manter a hierarquia das variáveis
df['parent_education_level'] = df['parent_education_level'].map({'High School': 0, 'Bachelor': 1, 'Master': 2})
df['study_environment'] = df['study_environment'].map({'Noisy': 0, 'Moderate': 1, 'Quiet': 2})

# Removemos variáveis que causam vazamento no alvo
df.drop(columns=['grade', 'pass_fail'], inplace=True) # 'inplace=True' indica que a operação será aplicada diretamente em df

Note que retiramos as variáveis "grade" e "pass_fail", pois elas não são as causas da nota final de um aluno, elas são na verdade as consequências diretas dela. Ou seja, se mantermos elas, o algoritmo é treinado com informações do futuro, criando uma trapaça matemática (também chamado de data leakage)

### Separação de treino e teste

Aqui, vamos pegar os dados e separar em dois: o grupo de **treino** e o grupo de **teste**.

A ideia é que no grupo de treino (geralmente 70% a 80% dos dados) entregamos as variavéis e as respostas finais para a máquina, com o intuito de ela estudar e encontrar os padrões.
No grupo de teste (os 20% a 30% restantes), escondemos as respostas finais da máquina, entregamos apenas as características dos alunos e mandamos ela adivinhar a variável preditora. Depois, comparamos os palpites dela com o gabarito real para calcular a "nota" de acerto do modelo (acurácia, erro médio, etc).

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['final_score']) # características (matriz com todas as colunas de variáveis, exceto o alvo)
y = df['final_score']                # alvo (nota final, o vetor que queremos prever)

# Corta a base de dados em dois pedaços (treino/teste) aleatórios
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=50) # 30% de dados de teste, 70% para treino

Perceba que a função *train_test_split* retorna 4 coisas:
* X_treino (as pistas de 70% dos alunos)
* y_treino (as notas finais desses 70%)
* X_teste (as pistas dos 30% restantes)
* y_teste (as notas reais desses 30%)

*Obs:* 'random_state' é o corte aleatório na base de dados. como o corte é aleatório, se não usarmos isso, toda vez que você que o código for rodado, ele vai separar alunos diferentes e os resultados vão ficar mudando. então, definindo um número qualquer, isso garante que o corte aleatório seja sempre o mesmo.


### Escalonamento

Nessa etapa, vamos espremer todas as colunas para a **mesma escala**, para que a máquina avalie de forma justa, sem achar uma variável mais importante que outra só porque o número é maior.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Criamos o escalonador
scaler = StandardScaler()

# A máquina aprende as proporções (fit) apenas com os dados de treino e as aplica (transform)
X_treino_escalonado = scaler.fit_transform(X_treino)

# Ela aplica (transform) o que aprendeu nos dados de teste
X_teste_escalonado = scaler.transform(X_teste)

Para padronizar os dados, o algoritmo executa duas etapas matemáticas:
* **fit** (calcular/ajustar): a máquina lê os dados e extrai os parâmetros estatísticos (média e desvio padrão). É aqui que ela molda a régua de medição.
* **transform** (aplicar/transformar): a máquina utiliza os parâmetros recém-calculados para alterar matematicamente os números do banco de dados, colocando-os na nova escala.

No nosso caso, note que, nos dados de treino, o escalonador calcula a régua baseada nos dados do passado e já aplica a transformação. Assim, nos dados de teste, aplicamos somente a transformação.

Isso porque, a máquina é obrigada a **avaliar os dados novos usando estritamente a régua que aprendeu no treino**. Se usássemos o fit no teste, a média seria recalculada, o que equivale a "espiar o futuro". Isso causa data leakage e arruína a validade do modelo para o mundo real.

### Treinamento

Essa é a fase em que a máquina estuda o passado para conseguir prever o futuro. Entregamos ao algoritmo o histórico de comportamento dos alunos e as suas respectivas notas finais. A partir dai, a máquina faz cálculos para descobrir qual é a fórmula matemática exata (o padrão oculto) que conecta o esforço do aluno à nota que ele tira. Uma vez aprendida essa regra, o modelo consegue receber os dados de um aluno totalmente novo e prever a sua nota.

Como o nosso objetivo é adivinhar a nota final (um valor numérico contínuo), este é um problema estatístico de **regressão**. Para resolvê-lo, testamos duas abordagens com lógicas de pensamento diferentes para ver qual se sai melhor:

> **Regressão linear múltipla**: esse modelo é excelente para encontrar relações de proporção direta. Ele tenta traçar uma linha reta matemática que resuma o comportamento geral da turma

> **Árvore de decisão (Decision tree)**: este modelo pensa de forma diferente, criando um fluxograma de perguntas lógicas. É ideal para captar padrões mais complexos e quebras de expectativa que não seguem uma linha reta


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# Modelo de regressão linear
modelo_linear = LinearRegression()                            # criamos o cérebro da máquina
modelo_linear.fit(X_treino_escalonado, y_treino)              # treinamos o modelo com os dados de treino + respostas do treino
previsoes_linear = modelo_linear.predict(X_teste_escalonado)  # previsões do modelo com os dados de teste

# Modelo de árvore de decisão
modelo_arvore = DecisionTreeRegressor(random_state=50, max_depth=5)   # criamos o cérebro da máquina
modelo_arvore.fit(X_treino_escalonado, y_treino)                      # treinamos o modelo com os dados de treino + respostas do treino
previsoes_arvore = modelo_arvore.predict(X_teste_escalonado)          # previsões do modelo com os dados de teste

*Obs:* a árvore de decisão funciona fazendo perguntas (ex: faltou muito? estuda pouco?). Então, deixamos o parâmetro '*max_depth*' igual a 5, pois, se deixarmos ela fazer infinitas perguntas, ela vai criar um ramo exclusivo para cada aluno e decorar o banco de dados inteiro. Colocando 5, fazemos com que ela faça no máximo 5 perguntas seguidas, obrigando a árvore a aprender regras gerais, e não a decorar.

### Resultados

Para finalizar e saber se a máquina foi bem, pegamos o vetor 'y_teste' (nosso gabarito) e cruzamos com os palpites que a máquina deu ('previsoes_linear' e 'previsoes_arvore').

Para isso, vamos usar a função *r2_score*, que roda uma fórmula matemática complexa e devolve um número de 0 a 1. Esse número indica a porcentagem das variáveis respostas (no nosso caso, final_score) que podem ser explicadas pelas variáveis que usados.

Além disso, também vamos usar a função *mean_absolute_error*, que pega o erro de cada palpite, transforma tudo em positivo (absoluto) e tira a média. Esse número N significa que, em média, a máquina errou a previsão da variável resposta em N unidades (para cima ou para baixo).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Calcula o R² de cada modelo
r2_linear = r2_score(y_teste, previsoes_linear)
r2_arvore = r2_score(y_teste, previsoes_arvore)

# Calcula o Erro Médio Absoluto (MAE) de cada modelo
erroMedioAbsoluto_linear = mean_absolute_error(y_teste, previsoes_linear)
erroMedioAbsoluto_arvore = mean_absolute_error(y_teste, previsoes_arvore)

print("--- MODELO 1: Regressão Linear Múltipla ---")
print(f"R²: {r2_linear:.4f}")
print(f"Erro médio absoluto: {erroMedioAbsoluto_linear:.2f} pontos\n")

print("--- MODELO 2: Árvore de Decisão ---")
print(f"R²: {r2_arvore:.4f}")
print(f"Erro médio absoluto: {erroMedioAbsoluto_arvore:.2f} pontos\n")

--- MODELO 1: Regressão Linear Múltipla ---
R²: 0.8795
Erro médio absoluto: 4.01 pontos

--- MODELO 2: Árvore de Decisão ---
R²: 0.7584
Erro médio absoluto: 5.58 pontos



### Conclusão

Após testar os dois algoritmos, o modelo de Regressão Linear Múltipla provou ser a melhor ferramenta preditiva nesse caso, superando a Árvore de Decisão nas métricas.

O modelo de regressão teve uma **alta precisão** (R² de 0.8795), conseguindo explicar praticamente 88% do comportamento das notas finais com base nas variáveis do alunos; e uma **baixa margem de erro** (MAE de 4.01), errando, em média, por 4 pontos da verdadeira nota (em uma escala de 0 a 100).

Portanto, sim, **é totalmente possível prever a nota final**. A regressão linear nos prova estatisticamente que as notas não são aleatórias, mas sim o resultado direto e proporcional das diferentes variáveis do aluno.